# Exercice 1.1 : l'agent de support en `StateGraph` (45 min, J1 15h30–16h15)

> Prérequis : notebooks J1-02 et J1-03. Ce notebook est **autonome** : exécutez-le de haut en bas.

**Objectif :** reconstruire l'agent support N1 de NordTech SA sous forme de graphe explicite, tracé dans Langfuse et protégé contre les boucles.

| Étape | Consigne |
|---|---|
| 1. Outils | `statut_serveur(nom)` et `chercher_kb(question)`, avec des docstrings **précises** et `parse_docstring=True` |
| 2. Graphe | `START → agent ⇄ tools → END`, avec `MessagesState`, `ToolNode` et `tools_condition` |
| 3. Prompt système | rôle N1 NordTech, réponses en français, **aucune invention** |
| 4. Tracing | session `ex11-<vos initiales>`, tags `stategraph` et `ex11` |
| 5. Robustesse | `recursion_limit=10` et gestion de `GraphRecursionError` (réponse de repli polie) |
| 6. Test | les 3 questions fournies, dont une qui nécessite **deux** outils |

**Critères de réussite :** le graphe compile et s'affiche · chaque question laisse une trace dans la session · la question 3 déclenche **deux** outils · aucune réponse n'invente un serveur ou une procédure.

## 0. Mise en route (fourni)

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
# Données fictives de NordTech SA (src/helpdesk_ops/donnees.py) : à lire pour savoir ce que les outils peuvent trouver
from helpdesk_ops.donnees import INVENTAIRE, KB

print(f"{len(INVENTAIRE)} serveurs : {', '.join(INVENTAIRE)}")
for article in KB:
    print(f"  [{article['id']}] {article['titre']}")

## 1. Les outils

Complétez les docstrings et le corps des fonctions. Le LLM ne lira **que** le nom, la docstring et le schéma des arguments.

*Indice pour `chercher_kb` :* une recherche par mots communs entre la question et `titre + contenu` suffit (pensez à `sans_accents` et à `.lower()`). Renvoyez au plus 2 articles, ou un message explicite si rien ne correspond.

In [ ]:
from langchain_core.tools import tool


@tool(parse_docstring=True)
def statut_serveur(nom: str) -> str:
    """TODO : que fait l'outil ? (une phrase précise)

    Args:
        nom: TODO : format attendu, avec un exemple
    """
    ...  # TODO


@tool(parse_docstring=True)
def chercher_kb(question: str) -> str:
    """TODO

    Args:
        question: TODO
    """
    ...  # TODO


# Testez vos outils SANS LLM avant de les donner à l'agent
# print(statut_serveur.invoke({"nom": "srv-vpn-01"}))
# print(chercher_kb.invoke({"question": "configurer le VPN"}))

## 2. Le graphe

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

OUTILS = [statut_serveur, chercher_kb]
llm_outils = llm.bind_tools(OUTILS)

PROMPT_SYSTEME = SystemMessage("TODO : rôle, langue, règles (aucune invention)")


def agent(state: MessagesState) -> dict:
    ...  # TODO : appeler llm_outils avec le prompt système + l'historique


# TODO : construire, câbler et compiler le graphe dans `support`
support = ...

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f"```mermaid\n{support.get_graph().draw_mermaid()}\n```"))

## 3. Une fonction `repondre` tracée et bornée

In [ ]:
from langgraph.errors import GraphRecursionError

INITIALES = "xx"   # TODO : vos initiales


def repondre(question: str) -> str:
    """Pose une question à l'agent ; ne lève jamais d'exception vers l'utilisateur."""
    ...  # TODO : invoke avec callbacks, run_name, recursion_limit=10 ; repli si GraphRecursionError

## 4. Les tests

In [ ]:
QUESTIONS = [
    "Le serveur srv-erp-02 est-il en ligne ?",
    "Outlook plante dès que j'ouvre une pièce jointe, que faire ?",
    "Mon VPN ne marche plus : srv-vpn-01 est-il en ligne, et comment reconfigurer le client ?",   # 2 outils
]

with propagate_attributes(session_id=f"ex11-{INITIALES}", tags=["stategraph", "ex11"]):
    for q in QUESTIONS:
        print(f"❓ {q}\n💬 {repondre(q)}\n")
langfuse.flush()

## 5. Dans Langfuse

1. **Sessions → `ex11-<initiales>`** : 3 traces ?
2. Question 3 : combien d'appels d'outils ? Dans quel ordre ? En un ou deux tours ?
3. Une réponse contient-elle une information **absente** des outils ? Si oui, durcissez le prompt système et relancez.